<a href="https://colab.research.google.com/github/AR-Ashik-9997/Phitron-practice-problem/blob/main/CNN_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [92]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader,Subset
from PIL import Image
import kagglehub

In [93]:
torch.manual_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(f"using Device {device}")

using Device cuda


In [94]:
# Download Dataset
path=kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path",path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path /kaggle/input/plantvillage


In [95]:
TRAIN_PATH=os.path.join(path,"PlantVillage","train")
VAL_PATH=os.path.join(path,"PlantVillage","val")

In [96]:
transforms=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3,std=[0.5]*3)
])

In [97]:
class MultiClassClassification(Dataset):
  def __init__(self,root_dir,transforms=None):
    super().__init__()

    self.samples=[]
    self.transforms=transforms
    self.classes=sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])
    self.class_to_idx={cls_name: idx for idx,cls_name in enumerate(self.classes)}

    self.samples = [
        (os.path.join(root_dir, c, image_path), self.class_to_idx[c])
        for c in self.classes
        for image_path in os.listdir(os.path.join(root_dir, c))
        if os.path.isfile(os.path.join(root_dir, c, image_path))
        ]

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    img_path,label=self.samples[idx]
    image=Image.open(img_path).convert("RGB")

    if self.transforms:
      image=self.transforms(image)
    return image,label

In [98]:
train_dataset_full=MultiClassClassification(TRAIN_PATH,transforms)
test_dataset_full=MultiClassClassification(VAL_PATH,transforms)
num_classes=len(train_dataset_full.classes)

print(f"Num_class:{num_classes}")
print(f"full_train_size:{len(train_dataset_full)}")
print(f"full_Test_size:{len(test_dataset_full)}")

Num_class:38
full_train_size:43444
full_Test_size:10861


In [99]:
train_dataset=Subset(train_dataset_full,list(range(100,len(train_dataset_full))))
test_dataset=Subset(test_dataset_full,list(range(100,len(test_dataset_full))))

In [100]:
pin=True if device.type=='cuda' else False
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=pin)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True,pin_memory=pin)